# ***MODELAGEM***

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import warnings

from scipy.integrate import solve_ivp
from scipy.signal import StateSpace, lsim, tf2ss

import sympy as sp
from sympy import symbols, Matrix, eye, simplify, factor, pprint, solve, Poly

import control
from control import ctrb, tf

In [ ]:
# TODO: reorganizar e rodar
# Cores Interessantes:
laranja = '#ff7f0e'
azul = '#1f77b4'

ciano_neon = '#00FFFF'
laranja_neon = '#FF5F1F'
amarelo_acido = '#FFFF33'
vermelho_neon = '#FF073A'
magenta = '#FF00FF'
verde_limao = '#39FF14'
roxo_eletrico = '#BF00FF'

In [ ]:
########################## Parâmetros do Sistema ###############################
# Parâmetros:
rho = 1025 # massa específica da água (kg/m3)
CDl = 0.01 # coeficiente de arrasto dos lemes (fólio fino) (adimensional)
CDc = 1.0 # coeficiente de arrasto 2D do casco (cilindro) (adimensional)
c = 0.026 # corda do fólio (m)
b = 0.052 # envergadura do fólio (m)
m = 16.0 # massa do AUV (kg)
D = 0.15 # diâmetro do AUV (m)
L = 1.2 # comprimento do AUV (m)
GB = D/4 # distância entre G e B (m)
Jyy = (m/2)*(1/12)*(3*((D/2)**2)+(L**2)) + (m/2)*((D/2)**2) # momento de inércia transversal do cilíndro em relação a B (kg.m2)
Jzz = m*(1/12)*(3*((D/2)**2)+(L**2)) # momento de inércia transversal do cilíndro em relação a B (kg.m2)
g = 9.81 # aceleração gravitacional (m/s2)
d = 0.9 *(L/2) # distância do centro do fólio em relação à meia nau (m)
U = 2.0 # velocidade horizontal inicial do AUV (m/s)
m11 = 0.029*rho*math.pi*(D**2)*L*(1/6) # massa adicional de surge (kg)
m33 = 0.96*rho*math.pi*(D**2)*L*(1/4) # massa adicional de heave (kg)
m22 = m33 # massa adcional de sway (kg)
m55 = 0.96*rho*math.pi*(L**3)*(1/12)*(D**2) # momento de inércia adicional de pitch (kg.m2)
m66 = m55 # momento de inércia adicional de yaw (kg.m2)
omega_m = math.radians(10)/1 # velocidade angular máxima de pitch (rad/s)
theta_m = math.radians(15) # ângulo máximo de pitch (graus)
defl = -10 # ângulo de deflexão dos lemes (teste degrau) (graus)
ta = 10 # tempo inicial de atuação do leme (s)
tb = 40 # tempo final de atuação do leme (s)

########################## Funções Auxiliares ##################################
# Sustentação dos Fólios:
def Lift_foil(u, w, alpha):
  CL = 2*math.pi*np.sin(alpha)
  return (1/2)*rho*c*(u**2 + w**2)*b*CL

# Arrasto dos Fólios:
def Drag_foil(u, w):
  return (1/2)*rho*c*CDl*(u**2 + w**2)*b

# Sustentação no Casco:
def Lift_hull(u, w, beta):
  CL0 = 2*math.pi*np.sin(beta)
  CL = CL0 / (1 + (CL0/(math.pi*((D**2)/D*L)*0.8))) # correção 3D
  return (1/2)*rho*L*(u**2 + w**2)*D*CL

# Deflexão no Leme:
def phi_foil(t):
  if t >= ta and t <= tb:
    return math.radians(defl)
  else:
    return 0

######################### Equações de Movimento do LAUV ########################
# Sistema Não Linear:
def LAUVv_non_linear(t, X): # equações no plano vertical
    u, w, theta, theta_dot = X

    # ângulos
    beta = np.arctan2(w, u)
    phi = phi_foil(t)
    alpha = phi - beta

    # forças no fólio
    Lf = Lift_foil(u, w, alpha)
    Rf = Drag_foil(u, w)
    Lc = Lift_hull(u, w, beta)

    # Equações (resolvidas para derivadas)
    A = np.array([
        [m, 0, -m*GB],
        [0, m + m33, 0],
        [-GB*m, 0, (Jyy + m55)]
    ])
    b = np.array([
        -m*w*theta_dot,
        2*Lf*np.cos(beta) - 2*Rf*np.sin(beta) + (m+m11)*u*theta_dot - m*GB*theta_dot**2 - (1/2)*rho*0.82*CDc*D*L*w*np.abs(w) - Lc*np.cos(beta),
        - m*g*GB*np.sin(theta) + 2*(Lf*np.cos(beta) - Rf*np.sin(beta))*d + m*GB*w*theta_dot - 50*(1/32)*rho*0.80*CDc*D*(L**4)*theta_dot*np.abs(theta_dot) + Lc*(L/4)*np.cos(beta)
    ])
    # Resolve u_dot, w_dot, theta_2dot
    du_dt, dw_dt, dtheta_dot_dt = np.linalg.solve(A, b)

    return [du_dt, dw_dt, theta_dot, dtheta_dot_dt]

def LAUVh_non_linear(t, X): # equações no plano horizontal
    u, v, psi, psi_dot = X

    # ângulos
    beta = np.arctan2(v, u)
    phi = phi_foil(t)
    alpha = phi - beta

    # forças no fólio
    Lf = Lift_foil(u, v, alpha)
    Rf = Drag_foil(u, v)
    Lc = Lift_hull(u, v, beta)

    # Equações (resolvidas para derivadas)
    A = np.array([
        [m, 0, 0],
        [0, m + m22, 0],
        [0, 0, (Jzz + m66)]
    ])
    b = np.array([
        m*v*psi_dot,
        2*Lf*np.cos(beta) - 2*Rf*np.sin(beta) - (m+m11)*u*psi_dot - (1/2)*rho*0.82*CDc*D*L*v*np.abs(v) - Lc*np.cos(beta),
        - 2*(Lf*np.cos(beta) - Rf*np.sin(beta))*d - 50*(1/32)*rho*0.80*CDc*D*(L**4)*psi_dot*np.abs(psi_dot) - Lc*(L/4)*np.cos(beta)
    ])
    # Resolve u_dot, w_dot, theta_2dot
    du_dt, dw_dt, dpsi_dot_dt = np.linalg.solve(A, b)

    return [du_dt, dw_dt, psi_dot, dpsi_dot_dt]

In [ ]:
# Sistema Linear:
Av = np.zeros((4, 4))
Bv = np.zeros((4, 1))
Cv = np.zeros((3, 4))
Ddv = np.zeros((3, 1))

Av[0, 0] = 0
Av[0, 1] = GB*((- 2*rho*c*b*U*math.pi*d - 2*0.5*rho*CDl*c*b*U*d + 0.25*rho*(L**2)*D*U*math.pi)/(Jyy + m55 - m*(GB**2)))
Av[0, 2] = -(m*g*(GB**2))/(Jyy + m55 - m*(GB**2))
Av[0, 3] = -((omega_m*GB*rho*0.80*CDc*D*(L**4))/(Jyy + m55 - m*(GB**2)))*(50*(3/4)*(1/32))

Av[1, 0] = 0
Av[1, 1] = -(((3/8)*U*math.sin(theta_m)*rho*0.82*CDc*D*L + 2*rho*c*U*math.pi*b + 2*0.5*rho*CDl*c*b*U + rho*L*D*U*math.pi)/(m + m33))
Av[1, 2] = 0
Av[1, 3] = U*((m + m11)/(m + m33))

Av[2, 0] = 0
Av[2, 1] = 0
Av[2, 2] = 0
Av[2, 3] = 1

Av[3, 0] = 0
Av[3, 1] = (- 2*rho*c*b*U*math.pi*d - 2*0.5*rho*CDl*c*b*U*d + 0.25*rho*(L**2)*D*U*math.pi)/(Jyy + m55 - m*(GB**2))
Av[3, 2] = -(m*g*GB)/(Jyy + m55 - m*(GB**2))
Av[3, 3] = -((omega_m*rho*0.80*CDc*D*(L**4))/(Jyy + m55 - m*(GB**2)))*(50*(3/4)*(1/32))

Bv[0, 0] = (2*rho*c*b*(U**2)*math.pi*d*GB)/(Jyy + m55 - m*(GB**2))
Bv[1, 0] = (2*rho*c*(U**2)*math.pi*b)/(m + m33)
Bv[2, 0] = 0
Bv[3, 0] = (2*rho*c*b*(U**2)*math.pi*d)/(Jyy + m55 - m*(GB**2))

Cv[0, 0] = 1
Cv[0, 1] = 0
Cv[0, 2] = 0
Cv[0, 3] = 0

Cv[1, 0] = 0
Cv[1, 1] = 1
Cv[1, 2] = 0
Cv[1, 3] = 0

Cv[2, 0] = 0
Cv[2, 1] = 0
Cv[2, 2] = 1
Cv[2, 3] = 0

LAUVv_linear = StateSpace(Av, Bv, Cv, Ddv) # cria o sistema no Espaço de Estados (movimento no plano vertical)

Ah = np.zeros((4, 4))
Bh = np.zeros((4, 1))
Ch = np.zeros((3, 4))
Ddh = np.zeros((3, 1))

Ah[1, 0] = 0
Ah[1, 1] = -(((3/8)*U*math.sin(theta_m)*rho*0.82*CDc*D*L + 2*rho*c*U*math.pi*b + 2*0.5*rho*CDl*c*b*U + rho*L*D*U*math.pi)/(m + m22))
Ah[1, 2] = 0
Ah[1, 3] = -U*((m + m11)/(m + m22))

Ah[2, 0] = 0
Ah[2, 1] = 0
Ah[2, 2] = 0
Ah[2, 3] = 1

Ah[3, 0] = 0
Ah[3, 1] = (2*rho*c*b*U*math.pi*d + 2*0.5*rho*CDl*c*b*U*d - 0.25*rho*(L**2)*D*U*math.pi)/(Jzz + m66)
Ah[3, 2] = 0
Ah[3, 3] = -((omega_m*rho*0.80*CDc*D*(L**4))/(Jzz + m66))*(50*(3/4)*(1/32))

Bh[0, 0] = 0
Bh[1, 0] = (2*rho*c*(U**2)*math.pi*b)/(m + m33)
Bh[2, 0] = 0
Bh[3, 0] = -(2*rho*c*b*(U**2)*math.pi*d)/(Jzz + m66)

Ch[0, 0] = 1
Ch[0, 1] = 0
Ch[0, 2] = 0
Ch[0, 3] = 0

Ch[1, 0] = 0
Ch[1, 1] = 1
Ch[1, 2] = 0
Ch[1, 3] = 0

Ch[2, 0] = 0
Ch[2, 1] = 0
Ch[2, 2] = 1
Ch[2, 3] = 0

LAUVh_linear = StateSpace(Ah, Bh, Ch, Ddh) # cria o sistema no Espaço de Estados (movimento no plano horizontal)

######################### Simulação do Sistema no Tempo ########################
# Tempo:
x0 = [U, 0.0, 0.0, 0.0]  # velocidades ângulo e velocidade angular iniciais (para ambos os movimentos)
t_span = (0, 80)         # tempo de simulação
t_eval = np.linspace(t_span[0], t_span[-1], 1000)

phi = math.radians(defl)*((t_eval >= ta) & (t_eval <= tb)).astype(float) # entrada degrau

# Resolução:
sol_v = solve_ivp(LAUVv_non_linear, t_span, x0, t_eval=t_eval, method='RK45') # simulação do movimento no plano vertical (não linear)
sol_h = solve_ivp(LAUVh_non_linear, t_span, x0, t_eval=t_eval, method='RK45') # simulação do movimento no plano horizontal (não linear)
tv_out, yv_out, xv_out = lsim(LAUVv_linear, U=phi, T=t_eval, X0=x0) # simulação do movimento no plano vertical (linear)
th_out, yh_out, xh_out = lsim(LAUVh_linear, U=phi, T=t_eval, X0=x0) # simulação do movimento no plano horizontal (linear)

# Extrair Soluções:
u_non_lin_v = sol_v.y[0] # movimento no plano vertical
w_non_lin = sol_v.y[1]
theta_non_lin = sol_v.y[2]

u_lin_v = yv_out[:,0]
w_lin = yv_out[:,1]
theta_lin = yv_out[:,2]

u_non_lin_h = sol_h.y[0] # movimento do plano horizontal
v_non_lin = sol_h.y[1]
psi_non_lin = sol_h.y[2]

u_lin_h = yh_out[:,0]
v_lin = yh_out[:,1]
psi_lin = yh_out[:,2]

# Converter para a Base Fixa:
U_non_lin_v = u_non_lin_v * np.cos(theta_non_lin) + w_non_lin * np.sin(theta_non_lin) # movimento no plano vertical
W_non_lin = - u_non_lin_v * np.sin(theta_non_lin) + w_non_lin * np.cos(theta_non_lin)

U_lin_v = u_lin_v * np.cos(theta_lin) + w_lin * np.sin(theta_lin)
W_lin = - u_lin_v * np.sin(theta_lin) + w_lin * np.cos(theta_lin)

#U_lin_v = u_lin_v - w_lin * theta_lin
#W_lin = u_lin_v * theta_lin + w_lin

U_non_lin_h = u_non_lin_h * np.cos(psi_non_lin) - v_non_lin * np.sin(psi_non_lin) # movimento no plano horizontal
V_non_lin = u_non_lin_h * np.sin(psi_non_lin) + v_non_lin * np.cos(psi_non_lin)

U_lin_h = u_lin_h * np.cos(psi_lin) - v_lin * np.sin(psi_lin)
V_lin = u_lin_h * np.sin(psi_lin) + v_lin * np.cos(psi_lin)

#U_lin_h = u_lin_h - v_lin * psi_lin
#V_lin = u_lin_h * psi_lin + v_lin

# Posições:
X0 = 0 # posição longitudinal inicial (m)
Y0 = 0 # posição transversal inicial (m)
Z0 = -100 # cota vertical inicial (m)
dt = sol_v.t[1] - sol_v.t[0]

X_non_lin_v = X0 + np.cumsum(U_non_lin_v) * dt # movimento no plano vertical
Z_non_lin = Z0 + np.cumsum(W_non_lin) * dt

X_lin_v = X0 + np.cumsum(U_lin_v) * dt
Z_lin = Z0 + np.cumsum(W_lin) * dt

X_non_lin_h = X0 + np.cumsum(U_non_lin_h) * dt # movimento no plano horizontal
Y_non_lin = Y0 + np.cumsum(V_non_lin) * dt

X_lin_h = X0 + np.cumsum(U_lin_h) * dt
Y_lin = Y0 + np.cumsum(V_lin) * dt

############################ Plotagem dos Resultados ###########################
# Posições:
plt.figure(figsize=(10,8))

plt.subplot(2,1,2)
plt.plot(X_non_lin_v, Z_non_lin, color=verde_limao, linestyle='-', label='Modelo Não Linear')
plt.plot(X_lin_v, Z_lin, color=verde_limao, linestyle='--', label='Modelo Linear')
plt.legend(loc="center right")
plt.xlabel('Posição X (m)')
plt.ylabel('Posição Z (m)')
plt.grid(True)
plt.title(f'Trajetória do LAUV no Plano Vertical - Teste de Leme: $\phi$ = {defl:.1f} graus; [{ta:.0f},{tb:.0f}] s', fontsize=14)

plt.tight_layout()
plt.show()

plt.figure(figsize=(10,8))

plt.subplot(2,1,2)
plt.plot(X_non_lin_h, Y_non_lin, color=verde_limao, linestyle='-', label='Modelo Não Linear')
plt.plot(X_lin_h, Y_lin, color=verde_limao, linestyle='--', label='Modelo Linear')
plt.legend(loc="lower right")
plt.xlabel('Posição X (m)')
plt.ylabel('Posição Y (m)')
plt.grid(True)
plt.title(f'Tragetória do LAUV no Plano Horizontal - Teste de Leme: $\phi$ = {defl:.1f} graus; [{ta:.0f},{tb:.0f}] s', fontsize=14)

plt.tight_layout()
plt.show()

# Velocidades:
plt.figure(figsize=(10,8))

plt.subplot(2,1,2)
plt.plot(sol_v.t, U_non_lin_v, color=ciano_neon, linestyle='-', label='surge (modelo não linear)')
plt.plot(sol_v.t, U_lin_v, color=ciano_neon, linestyle='--', label='surge (modelo linear)')
plt.plot(sol_v.t, W_non_lin, color=laranja_neon, linestyle='-', label='heave (modelo não linear)')
plt.plot(sol_v.t, W_lin, color=laranja_neon, linestyle='--', label='heave (modelo linear)')
plt.legend(loc="center right")
plt.xlabel('Tempo (s)')
plt.ylabel('Velocidades (m/s)')
plt.grid(True)
plt.title(f'Velocidades do LAUV no Plano Vertical - Teste de Leme: $\phi$ = {defl:.1f} graus; [{ta:.0f},{tb:.0f}] s', fontsize=14)

plt.tight_layout()
plt.show()

plt.figure(figsize=(10,8))

plt.subplot(2,1,2)
plt.plot(sol_h.t, U_non_lin_h, color=ciano_neon, linestyle='-', label='surge (modelo não linear)')
plt.plot(sol_h.t, U_lin_h, color=ciano_neon, linestyle='--', label='surge (modelo linear)')
plt.plot(sol_h.t, V_non_lin, color=laranja_neon, linestyle='-', label='sway (modelo não linear)')
plt.plot(sol_h.t, V_lin, color=laranja_neon, linestyle='--', label='sway (modelo linear)')
plt.legend(loc="upper right")
plt.xlabel('Tempo (s)')
plt.ylabel('Velocidades (m/s)')
plt.grid(True)
plt.title(f'Velocidades do LAUV no Plano Horizontal - Teste de Leme: $\phi$ = {defl:.1f} graus; [{ta:.0f},{tb:.0f}] s', fontsize=14)

plt.tight_layout()
plt.show()

# Ângulos:
plt.figure(figsize=(10,8))

plt.subplot(2,1,2)
plt.plot(sol_v.t, np.rad2deg(theta_non_lin), color=vermelho_neon, linestyle='-', label=r'$\theta$(t) - Modelo Não Linear')
plt.plot(sol_v.t, np.rad2deg(theta_lin), color=vermelho_neon, linestyle='--', label=r'$\theta$(t) - Modelo Linear')
plt.legend(loc="center right")
plt.xlabel('Tempo (s)')
plt.ylabel('Ângulo (graus)')
plt.grid(True)
plt.title(f'Ângulo de Pitch do LAUV - Movimento no Plano Vertical - Teste de Leme: $\phi$ = {defl:.1f} graus; [{ta:.0f},{tb:.0f}] s', fontsize=14)

plt.tight_layout()
plt.show()

#psi_360 = np.mod(np.degrees(psi_non_lin), 360)
plt.figure(figsize=(10,8))

plt.subplot(2,1,2)
plt.plot(sol_h.t, np.rad2deg(psi_non_lin), color=vermelho_neon, linestyle='-', label='$\psi$(t) - Modelo Não Linear')
plt.plot(sol_v.t, np.rad2deg(psi_lin), color=vermelho_neon, linestyle='--', label='$\psi$(t) - Modelo Linear')
plt.legend(loc="center right")
plt.xlabel('Tempo (s)')
plt.ylabel('Ângulo (graus)')
plt.grid(True)
plt.title(f'Ângulo de Yaw do LAUV - Movimento no Plano Horizontal - Teste de Leme: $\phi$ = {defl:.1f} graus; [{ta:.0f},{tb:.0f}] s', fontsize=14)

plt.tight_layout()
plt.show()

# ***CONTROLE - PLANO VERTICAL***

In [ ]:
########################## Parâmetros do Sistema ###############################
# Parâmetros:
rho = 1025 # massa específica da água (kg/m3)
CDl = 0.01 # coeficiente de arrasto dos lemes (fólio fino) (adimensional)
CDc = 1.0 # coeficiente de arrasto 2D do casco (cilindro) (adimensional)
c = 0.026 # corda do fólio (m)
b = 0.052 # envergadura do fólio (m)
m = 16.0 # massa do AUV (kg)
D = 0.15 # diâmetro do AUV (m)
L = 1.2 # comprimento do AUV (m)
GB = D/4 # distância entre G e B (m)
Jyy = (m/2)*(1/12)*(3*((D/2)**2)+(L**2)) + (m/2)*((D/2)**2) # momento de inércia transversal do cilíndro em relação a B (kg.m2)
Jzz = m*(1/12)*(3*((D/2)**2)+(L**2)) # momento de inércia transversal do cilíndro em relação a B (kg.m2)
g = 9.81 # aceleração gravitacional (m/s2)
d = 0.9 *(L/2) # distância do centro do fólio em relação à meia nau (m)
U = 2.0 # velocidade horizontal inicial do AUV (m/s)
m11 = 0.029*rho*math.pi*(D**2)*L*(1/6) # massa adicional de surge (kg)
m33 = 0.96*rho*math.pi*(D**2)*L*(1/4) # massa adicional de heave (kg)
m22 = m33 # massa adcional de sway (kg)
m55 = 0.96*rho*math.pi*(L**3)*(1/12)*(D**2) # momento de inércia adicional de pitch (kg.m2)
m66 = m55 # momento de inércia adicional de yaw (kg.m2)
omega_m = math.radians(10)/1 # velocidade angular máxima de pitch (rad/s)
theta_m = math.radians(15) # ângulo máximo de pitch (graus)
defl = -10 # ângulo de deflexão dos lemes (teste degrau) (graus)
ta = 10 # tempo inicial de atuação do leme (s)
tb = 40 # tempo final de atuação do leme (s)

############################ Configurações Iniciais ############################
# Condições Iniciais:
Time = [0, 80] # intervalo de simulação (s)
Uc = 2.0 # velocidade horizontal inicial (m/s)
X0 = np.array([Uc, 0, 0, 0])  # vetor de estado inicial X0 = [u, w, theta, theta_dot]
z0 = -100.0  # posição vertical inicial (m)
z_ref = -80.0  # referência desejada (m)

# Restrições:
phi_max = np.deg2rad(10.0)  # saturação do atuador
pitch_max = np.deg2rad(15) # ângulo máximo de pitch (simular um controle de pitch...)

In [ ]:
################################ Controlabilidade ##############################
# Verificação de Controlabilidade:
CO = ctrb(Av, Bv) # matriz de controlabilidade (sistema original)
print(f'\nA matriz de controlabilidade do sistema é:  CO = {CO}')
print(f'\nO posto da matriz CO é: {np.linalg.matrix_rank(CO)}')

# Sistema no Espaço de Estados Controlável:
Avc = np.zeros((4, 4))
Bvc = np.zeros((4, 1))

Avc[0, 0] = -(((3/8)*U*math.sin(theta_m)*rho*0.82*CDc*D*L + 2*rho*c*U*math.pi*b + 2*0.5*rho*CDl*c*b*U + rho*L*D*U*math.pi)/(m + m33))
Avc[0, 1] = 0
Avc[0, 2] = U*((m + m11)/(m + m33))
Avc[0, 3] = 0

Avc[1, 0] = 0
Avc[1, 1] = 0
Avc[1, 2] = 1
Avc[1, 3] = 0

Avc[2, 0] = (- 2*rho*c*b*U*math.pi*d - 2*0.5*rho*CDl*c*b*U*d + 0.25*rho*(L**2)*D*U*math.pi)/(Jyy + m55 - m*(GB**2))
Avc[2, 1] = -(m*g*GB)/(Jyy + m55 - m*(GB**2))
Avc[2, 2] = -((omega_m*rho*0.80*CDc*D*(L**4))/(Jyy + m55 - m*(GB**2)))*(100*(3/4)*(1/64))
Avc[2, 3] = 0

Avc[3,0] = 1
Avc[3,1] = -U
Avc[3, 2] = 0
Avc[3, 3] = 0

Bvc[0, 0] = (2*rho*c*(U**2)*math.pi*b)/(m + m33)
Bvc[1, 0] = 0
Bvc[2, 0] = (2*rho*c*b*(U**2)*math.pi*d)/(Jyy + m55 - m*(GB**2))
Bvc[3, 0] = 0

print(f'\n Matriz Av (ordem 4) = {Av}')
print(f'\nMatriz Bv (ordem 4) = {Bv}')

COc = ctrb(Avc, Bvc) # matriz de controlabilidade
print(f'\nA matriz de controlabilidade do sistema é:  COc = {COc}')
print(f'\nO posto da matriz COc é: {np.linalg.matrix_rank(COc)}')

print(f'\n Matriz Av controlável = {Avc}')
print(f'\nMatriz Bv controlável = {Bvc}')

# Estudo de Autovalores:
eigvals_v, eigvecs_v= np.linalg.eig(Av)
eigvals_vc, eigvecs_vc= np.linalg.eig(Avc)
print(f'\nOs autovalores de Av são: {eigvals_v}')
print(f'\nOs autovalores de Av controlável são: {eigvals_vc}')

In [ ]:

##################### Projeto do Controlador no Espaço de Estados ##############
# Alocação de Polos para Sistema com Polo na Origem:
wn = 1.8
zeta = 0.707
s1 = -wn*zeta + 1j*wn*np.sqrt(1 - zeta**0.5)
s2 = -wn*zeta - 1j*wn*np.sqrt(1 - zeta**0.5)
s3 = 5*(-wn*zeta)
s4 = 6*(-wn*zeta)
polos = [s1, s2, s3, s4] # polos desejados

Kcon = control.place(Avc, Bvc, polos) # alocação

print(f'\nOs polos alocados são: poles = {polos}')
print(f'\nA matriz de ganho de realimentação de estado é: K = {Kcon}')

# Sistema de Controle em Malha Fechada:
Av_con = Avc - Bvc @ Kcon
C = np.array([0,0,0,1])

# calcular kr de forma numericamente estável (usa solve ao invés de inv)
# calcula temp = (A-BK)^{-1} B  solucionando (A-BK) temp = B
temp = np.linalg.solve(Av_con, Bvc)   # (n,1)
denom = (C @ temp).item()       # escalar

if abs(denom) < 1e-9:
    raise RuntimeError("\nDenominação perto de zero: (C (A-BK)^(-1) B) ≈ 0. "
                       "Não é possível calcular kr diretamente. Considere integrar erro.")
kr = -1.0 / denom

print(f'\nO ganho kr é: {kr}\n')

################################# Modelo Linear ################################
# Sistema de Controle (Alocação) com Pitch Saturado:
def LAUVvc_lin_psat(t, X):
    X = X.reshape(-1, 1)          # (n,1)

    # Saturação de Pitch:
    theta_sat = np.clip(X[1],-pitch_max, pitch_max)
    X[1] = theta_sat

    # Atuador
    u = float(- (Kcon @ X) + kr * z_ref)   # u escalar
    u_sat = np.clip(u, -phi_max, phi_max)  # saturação...

    X_dot = Avc @ X + Bvc * u_sat        # (n,1)
    return X_dot.flatten()

# Sistema de Controle (Alocação):
def LAUVvc_lin(t, X):
    X = X.reshape(-1, 1)          # (n,1)

    # Atuador
    u = float(- (Kcon @ X) + kr * z_ref)   # u escalar
    u_sat = np.clip(u, -phi_max, phi_max)  # saturação...

    X_dot = Avc @ X + Bvc * u_sat        # (n,1)
    return X_dot.flatten()

############################## Modelo Não Linear ###############################
# Dinâmica do LAUV em Malha Fechada (Controle por Alocação) com Pitch Saturado:
def LAUVvc_non_lin_psat(t, X_aug):
  u, w, theta, theta_dot, z = X_aug
  X = np.hstack((w, theta, theta_dot, z))

  # saturação de pitch
  theta = np.clip(theta, -pitch_max, pitch_max)
  X_aug[2] = theta
  X[1] = theta

  # velocidade vertical absoluta
  W = - u * np.sin(theta) + w * np.cos(theta)  # z_dot

  # Controlador
  u_ctrl = float(- (Kcon @ X) + kr * z_ref)

  # saturação
  u_sat = np.clip(u_ctrl, -phi_max, phi_max)

  # ângulos
  beta = np.arctan2(w, u)
  phi = u_sat
  alpha = phi - beta

  # forças
  Lf = Lift_foil(u, w, alpha)
  Rf = Drag_foil(u, w)
  Lc = Lift_hull(u, w, beta)

  # dinâmica
  A = np.array([
      [m, 0, -m*GB],
      [0, m + m33, 0],
      [-GB*m, 0, (Jyy + m55)]
  ])
  b = np.array([
      -m*w*theta_dot,
      2*Lf*np.cos(beta) - 2*Rf*np.sin(beta) + (m+m11)*u*theta_dot - m*GB*theta_dot**2 - (1/2)*rho*0.82*CDc*D*L*w*np.abs(w) - Lc*np.cos(beta),
      - m*g*GB*np.sin(theta) + 2*(Lf*np.cos(beta) - Rf*np.sin(beta))*d + m*GB*w*theta_dot - 100*(1/64)*rho*0.80*CDc*D*(L**4)*theta_dot*np.abs(theta_dot) + Lc*(L/4)*np.cos(beta)
  ])

  du_dt, dw_dt, dtheta_dot_dt = np.linalg.solve(A, b)
  dtheta_dt = theta_dot

  X_dot = np.hstack((du_dt, dw_dt, dtheta_dt, dtheta_dot_dt))
  z_dot = W

  return np.hstack((X_dot, z_dot))

# Dinâmica do LAUV em Malha Fechada (Controle por Alocação):
def LAUVvc_non_lin(t, X_aug):
  u, w, theta, theta_dot, z = X_aug
  X = np.hstack((w, theta, theta_dot, z))

  # velocidade vertical absoluta
  W = - u * np.sin(theta) + w * np.cos(theta)  # z_dot

  # Controlador
  u_ctrl = float(- (Kcon @ X) + kr * z_ref)

  # saturação
  u_sat = np.clip(u_ctrl, -phi_max, phi_max)

  # ângulos
  beta = np.arctan2(w, u)
  phi = u_sat
  alpha = phi - beta

  # forças
  Lf = Lift_foil(u, w, alpha)
  Rf = Drag_foil(u, w)
  Lc = Lift_hull(u, w, beta)

  # dinâmica
  A = np.array([
      [m, 0, -m*GB],
      [0, m + m33, 0],
      [-GB*m, 0, (Jyy + m55)]
  ])
  b = np.array([
      -m*w*theta_dot,
      2*Lf*np.cos(beta) - 2*Rf*np.sin(beta) + (m+m11)*u*theta_dot - m*GB*theta_dot**2 - (1/2)*rho*0.82*CDc*D*L*w*np.abs(w) - Lc*np.cos(beta),
      - m*g*GB*np.sin(theta) + 2*(Lf*np.cos(beta) - Rf*np.sin(beta))*d + m*GB*w*theta_dot - 100*(1/64)*rho*0.80*CDc*D*(L**4)*theta_dot*np.abs(theta_dot) + Lc*(L/4)*np.cos(beta)
  ])

  du_dt, dw_dt, dtheta_dot_dt = np.linalg.solve(A, b)
  dtheta_dt = theta_dot

  X_dot = np.hstack((du_dt, dw_dt, dtheta_dt, dtheta_dot_dt))
  z_dot = W

  return np.hstack((X_dot, z_dot))

################################# Simulação ####################################
# Solver Numérico:
X0c = [0, 0, 0, z0] # condição inicial para vetor de estado para sistema controlável X0c = [w, theta, theta_dot, Z]
t_span = (Time[0], Time[1]) # intervalo de discretização
t_eval = np.linspace(*t_span, 2000) # intervalo de simulação
solc_lin_psat = solve_ivp(LAUVvc_lin_psat, t_span, X0c, t_eval=t_eval) # simulação linear com saturação no pitch
solc_lin = solve_ivp(LAUVvc_lin, t_span, X0c, t_eval=t_eval) # simulação linear sem saturação no pitch

# Extrair Soluções:
zc_lin_psat = solc_lin_psat.y[3]
thetac_lin_psat = solc_lin_psat.y[1]

zc_lin = solc_lin.y[3]
thetac_lin = solc_lin.y[1]

# Condições Inicias Ampliadas:
X0_aug = np.hstack((X0, [z0])) # concatenação de X0 com [z0]

# Solver Numérico:
t_span = (Time[0], Time[1]) # intervalo de discretização
t_eval = np.linspace(*t_span, 2000) # intervalo de simulação
solv_non_lin_psat = solve_ivp(LAUVvc_non_lin_psat, t_span, X0_aug, t_eval=t_eval, method='RK45') # simulação não linear com saturação de pitch
solv_non_lin = solve_ivp(LAUVvc_non_lin, t_span, X0_aug, t_eval=t_eval, method='RK45') # simulação não linear sem saturação de pitch

# Extrair Soluções:
u_non_lin_psat = solv_non_lin_psat.y[0]
w_non_lin_psat = solv_non_lin_psat.y[1]
theta_non_lin_psat = solv_non_lin_psat.y[2]
z_non_lin_psat = solv_non_lin_psat.y[4]

u_non_lin = solv_non_lin.y[0]
w_non_lin = solv_non_lin.y[1]
theta_non_lin = solv_non_lin.y[2]
z_non_lin = solv_non_lin.y[4]

# Atuador:
u_psat = np.zeros(solc_lin_psat.t.size)
for i in range(solc_lin_psat.t.size):
    X = solc_lin_psat.y[:, i]         # vetor de estado no instante t_i
    u_ctrl = float(-(Kcon @ X) + kr * z_ref)
    u_sat = np.clip(u_ctrl, -phi_max, phi_max)
    u_psat[i] = u_sat

u = np.zeros(solc_lin.t.size)
for i in range(solc_lin.t.size):
    X = solc_lin.y[:, i]         # vetor de estado no instante t_i
    u_ctrl = float(-(Kcon @ X) + kr * z_ref)
    u_sat = np.clip(u_ctrl, -phi_max, phi_max)
    u[i] = u_sat

########################## Plotagem dos Resultados #############################
# Solução com Saturação de Pitch:
# Posição Vertical:
plt.figure(figsize=(10,6))
plt.text(
    0.76, 0.40,                        # posição (em fração do eixo)
    f"$K_r$ = {kr:.2f}",
    transform=plt.gca().transAxes,     # posiciona relativo ao gráfico
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4')
)
plt.plot(solc_lin_psat.t, zc_lin_psat, color=ciano_neon, linestyle='--', label="Z(t): Posição Vertical do LAUV - Modelo Linear")
plt.plot(solv_non_lin_psat.t, z_non_lin_psat, color=ciano_neon, linestyle='-', label="Z(t): Posição Vertical do LAUV - Modelo Não Linear")
plt.axhline(z_ref, color='r', linestyle='--', label=f"cota de referência: Z = {z_ref:.1f} m")
plt.xlabel("Tempo (s)")
plt.ylabel("Posição Z (m)")
plt.title("Controlador de Cota Vertical do LAUV com Restrição de Pitch - Movimento no Plano Vertical")
plt.legend(loc="center right")
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# Ângulo de Pitch:
plt.figure(figsize=(10,6))
plt.text(
    0.75, 0.62,                        # posição (em fração do eixo)
    rf"$\theta_{{máx}}$ = {np.rad2deg(pitch_max):.1f} graus",
    transform=plt.gca().transAxes,     # posiciona relativo ao gráfico
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4')
)
plt.plot(solc_lin_psat.t, np.rad2deg(thetac_lin_psat), color=amarelo_acido, linestyle='--', label=r"$\theta$(t): Orientação Angular do LAUV - Modelo Linear")
plt.plot(solv_non_lin_psat.t, np.rad2deg(theta_non_lin_psat), color=amarelo_acido, linestyle='-', label=r"$\theta$(t): Orientação Angular do LAUV - Modelo Não Linear")
plt.xlabel("Tempo (s)")
plt.ylabel("Ângulo (Graus)")
plt.title("Ângulo de Pitch com Restrição - Controle no Espaço de Estado")
plt.legend(loc="center right")
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# Atuador:
plt.figure(figsize=(10,6))
plt.text(
    0.83, 0.90,                        # posição (em fração do eixo)
    f"$\phi_{{máx}}$ = {np.rad2deg(phi_max):.1f} graus",
    transform=plt.gca().transAxes,     # posiciona relativo ao gráfico
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4')
)
plt.plot(solc_lin_psat.t, np.rad2deg(u_psat), color=laranja_neon, linestyle='-', label="$\phi$(t): Ângulo do Leme")
plt.xlabel("Tempo (s)")
plt.ylabel("Ângulo (graus)")
plt.title("Ângulo de Deflexão do Leme Saturado - Pitch com Restrição")
plt.legend(loc="upper right")
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# Solução sem Saturação de Pitch:
# Posição Vertical:
plt.figure(figsize=(10,6))
plt.text(
    0.76, 0.40,                        # posição (em fração do eixo)
    f"$K_r$ = {kr:.2f}",
    transform=plt.gca().transAxes,     # posiciona relativo ao gráfico
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4')
)
plt.plot(solc_lin.t, zc_lin, color=ciano_neon, linestyle='--', label="Z(t): Posição Vertical do LAUV - Modelo Linear")
plt.plot(solv_non_lin.t, z_non_lin, color=ciano_neon, linestyle='-', label="Z(t): Posição Vertical do LAUV - Modelo Não Linear")
plt.axhline(z_ref, color='r', linestyle='--', label=f"cota de referência: Z = {z_ref:.1f} m")
plt.xlabel("Tempo (s)")
plt.ylabel("Posição Z (m)")
plt.title("Controlador de Cota Vertical do LAUV sem Restrição de Pitch - Movimento no Plano Vertical")
plt.legend(loc="center right")
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# Ângulo de Pitch:
plt.figure(figsize=(10,6))
plt.plot(solc_lin.t, np.rad2deg(thetac_lin), color=amarelo_acido, linestyle='--', label=r"$\theta$(t): Orientação Angular do LAUV - Modelo Linear")
plt.plot(solv_non_lin.t, np.rad2deg(theta_non_lin), color=amarelo_acido, linestyle='-', label=r"$\theta$(t): Orientação Angular do LAUV - Modelo Não Linear")
plt.xlabel("Tempo (s)")
plt.ylabel("Ângulo (Graus)")
plt.title("Ângulo de Pitch sem Restrição - Controle no Espaço de Estado")
plt.legend(loc="center right")
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# Atuador:
plt.figure(figsize=(10,6))
plt.text(
    0.83, 0.90,                        # posição (em fração do eixo)
    f"$\phi_{{máx}}$ = {np.rad2deg(phi_max):.1f} graus",
    transform=plt.gca().transAxes,     # posiciona relativo ao gráfico
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4')
)
plt.plot(solc_lin.t, np.rad2deg(u), color=laranja_neon, linestyle='-', label="$\phi$(t): Ângulo do Leme")
plt.xlabel("Tempo (s)")
plt.ylabel("Ângulo (graus)")
plt.title("Ângulo de Deflexão do Leme Saturado - Pitch sem Restrição")
plt.legend(loc="upper right")
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# **CONTROLE - PLANO HORIZONTAL**

In [ ]:
###################### Função de Transferência Analítica #######################
# Estudo de Autovalores:
eigvals, eigvecs = np.linalg.eig(Ah)
print("\nOs autovalores de Ah são:", eigvals)

print(f'\n Matriz Ah = {Ah}')
print(f'\nMatriz Bh = {Bh}')

R = Ah[3][3] + Ah[1][1]
Q = Ah[3][1]*Ah[1][3] - Ah[3][3]*Ah[1][1]

print(f'\nCoeficientes do polinômio característico: a3 = {-1}; a2 = {R:.1f}; a1 = {Q:.1f}; a0 = {0}')
print(f'\nCoeficientes do numerador: a1 = {-Bh[3][0]:.1f}; a0 = {(Bh[3][0]*Ah[1][1] - Ah[3][1]*Bh[1][0]):.1f}')

# --- Defina o símbolo de Laplace
s = symbols('s')

# --- Defina símbolos para constantes (exemplo)
ai,bi,ci,di,x,y= symbols('a b c d x y')

# --- Exemplo: matrizes 2x2, 2x1, 1x2 (altere conforme seu caso)
A = Matrix([[0, 0, 0, 0],
            [0, ci, 0, di],
            [0, 0, 0, 1],
            [0, ai, 0, bi]])
B = Matrix([[0],
            [x],
            [0],
            [y]])
C = Matrix([[0, 0, 1, 0]])
Dd = Matrix([[0]])  # se houver termo direto

# --- Matriz (sI - A)
sI_minus_A = s*eye(A.rows) - A

# --- Inversa simbólica
inv = sI_minus_A.inv()   # SymPy calcula simbolicamente se possível

# --- Função de transferência simbólica
G = simplify( C * inv * B + Dd )

# --- Melhor apresentação (fatorar polinômios onde fizer sentido)
G_factored = factor(G)

#print("Matriz (sI - A):")
#pprint(sI_minus_A)
#print("\nInversa (sI - A)^-1:")
#pprint(inv)
#print("\nG(s) = C (sI-A)^-1 B + D :")
#pprint(G)
print("\nG(s) fatorado:")
pprint(G_factored)

# Obter os polos para comparação:
s = symbols('s')
p = -s**3 + R*s**2 + Q*s

# Raízes simbólicas (exatas, se possível)
#roots = solve(p, s)
#print("Raízes simbólicas:", roots)

# Ou transformar em objeto polinômio e pedir raízes
poly = Poly(p, s)
roots_poly = poly.nroots()  # aproximação numérica (útil p/ coeficientes complicados)
print("\nRaízes numéricas da equação característica (comparação):", roots_poly,'\n')

############################ Lugar das Raízes ##################################
# =====================================================
# 1) SISTEMA, POLOS, ZEROS, INTERVALOS REAIS, BREAKPOINTS
# =====================================================

num = [1.6, 78.2]
den = [-1.0, -37.1, -117.5, 0.0]
G = tf(num, den)

poles = np.roots(den)
zeros = np.roots(num)

real_poles = np.real(poles[np.isclose(np.imag(poles), 0)])
real_zeros = np.real(zeros[np.isclose(np.imag(zeros), 0)])

# centróide e assíntotas
n = len(poles); m = len(zeros); r = n - m
centroid = (np.sum(poles) - np.sum(zeros)) / r
angles = [(2*k + 1)*np.pi/r for k in range(r)]

# -------- intervalos reais --------
pts = np.sort(np.concatenate((real_poles, real_zeros)))
intervals = []
for i in range(len(pts)+1):
    if i==0:
        a = -1e6; b = pts[0]
    elif i==len(pts):
        a = pts[-1]; b = 1e6
    else:
        a = pts[i-1]; b = pts[i]
    mid = 0.5*(a+b)
    count_right = np.sum(real_poles > mid) + np.sum(real_zeros > mid)
    if count_right % 2 == 1:
        intervals.append((a,b))

# -------- breakpoints (dK/ds=0) --------
s = sp.symbols('s')

D = -s**3 - 37.1*s**2 - 117.5*s
N = 1.6*s + 78.2
dD = sp.diff(D, s)
dN = sp.diff(N, s)

eq = sp.simplify(dD*N - D*dN)
sol = sp.solve(sp.Eq(eq, 0), s)

sol_real = []
for item in sol:
    v = complex(sp.N(item))
    if abs(v.imag) < 1e-8:
        sol_real.append(float(v.real))

sol_real = sorted(set([round(x,12) for x in sol_real]))

# -------- breakpoints filtrados --------
K_lower0 = -4359.25/18.84   # de Routh
K_lower = -1000 # plotagem...

valid_breakpoints = []
break_K = []
for bp in sol_real:
    if not any(a < bp < b for (a,b) in intervals):
        continue
    D_val = float(sp.N(D.subs(s,bp)))
    N_val = float(sp.N(N.subs(s,bp)))
    if abs(N_val) < 1e-12:
        continue
    Kval = -D_val/N_val
    if (np.isreal(Kval)) and (Kval < 0) and (Kval > K_lower):
        valid_breakpoints.append(bp)
        break_K.append(float(Kval))

# =====================================================
# 2) ROOT LOCUS NUMÉRICO — SEM root_locus()
# =====================================================

num_p = np.array([1.6, 78.2])
den_p = np.array([-1.0, -37.1, -117.5, 0.0])

deg_den = len(den_p)-1
deg_num = len(num_p)-1
num_pad = np.pad(num_p, (deg_den-deg_num,0))

def closed_loop_poles(K):
    coeff = den_p + K*num_pad
    if abs(coeff[0])<1e-12:
        coeff = np.trim_zeros(coeff, 'f')
    return np.roots(coeff)

Ks = np.linspace(K_lower*0.999, -1e-3, 1200)

# primeiro K
first = closed_loop_poles(Ks[0])
order = np.argsort(-np.real(first))
prev = list(first[order])
branches = [[z] for z in prev]

for K in Ks[1:]:
    cur = list(closed_loop_poles(K))
    assigned = [False]*len(cur)
    new = [None]*len(prev)

    for i, p in enumerate(prev):
        dists = [abs(p - c) if not assigned[j] else np.inf for j,c in enumerate(cur)]
        jmin = int(np.argmin(dists))
        assigned[jmin] = True
        new[i] = cur[jmin]

    for j,a in enumerate(assigned):
        if not a:
            # fallback simples
            for i in range(len(prev)):
                if new[i] is None:
                    new[i] = cur[j]
                    assigned[j] = True
                    break

    for i in range(len(prev)):
        branches[i].append(new[i])
    prev = new

branches = [np.array(b) for b in branches]

# =====================================================
# 3) PLOT FINAL — usando SEU estilo + RL numérico
# =====================================================

fig, ax = plt.subplots(figsize=(11,6))
ax.grid(True)
ax.set_title("Lugar das Raízes - Controlador de Rumo em Malha Fechada (Modelo Linear)")
ax.set_xlabel("Parte Real")
ax.set_ylabel("Parte Imaginária")

# ------ root locus numérico ------
'''
colors = plt.cm.viridis(np.linspace(0,1,len(branches)))
for col,c in zip(branches, colors):
    ax.plot(col.real, col.imag, '-', color=c, lw=1.8)
'''
colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(branches)))
for col, c in zip(branches, colors):
    ax.plot(col.real, col.imag, '-', color=c, lw=2.0)

# ------ polos e zeros ------
ax.plot(np.real(poles), np.imag(poles), 'x', markersize=10, color='cyan', label="polos")
ax.plot(np.real(zeros), np.imag(zeros), 'o', fillstyle='none', markersize=8,
        color='magenta', label="zeros")

# ------ centróide e assíntotas ------
ax.plot(np.real(centroid), 0, 's', color='yellow', markersize=8, label='centróide')
'''
xext = np.linspace(-200,200,400)
for ang in angles:
    if np.isclose(np.mod(ang, np.pi), np.pi/2):
        ax.plot([centroid.real, centroid.real], [-200,200], '--', color='gray')
    else:
        ax.plot(xext, np.tan(ang)*(xext - centroid.real), '--', color='gray')
'''
# assíntotas com legenda
for i, ang in enumerate(angles):
    if np.isclose(np.mod(ang, np.pi), np.pi/2):
        if i == 0:
            ax.plot([centroid.real, centroid.real], [-200, 200],
                    '--', color='gray', label='Assíntotas')
        else:
            ax.plot([centroid.real, centroid.real], [-200, 200],
                    '--', color='gray')
    else:
        if i == 0:
            ax.plot(xext, np.tan(ang)*(xext - centroid.real),
                    '--', color='gray', label='Assíntotas')
        else:
            ax.plot(xext, np.tan(ang)*(xext - centroid.real),
                    '--', color='gray')
# ------ intervalos reais do RL ------
for (a,b) in intervals:
    left = a if a > -1e5 else -80
    right = b if b < 1e5 else 20
    ax.plot([left,right], [0,0], color='red', lw=3)
'''
# ------ breakpoints ------
for bp, kv in zip(valid_breakpoints, break_K):
    ax.plot(bp, 0, 'go', markersize=10)
    ax.annotate(f"{bp:.3f}\nK={kv:.3g}", xy=(bp,0), xytext=(bp,6),
                ha='center', fontsize=9, color='white',
                bbox=dict(boxstyle="round,pad=0.2", fc="darkgreen", alpha=0.6))
'''
# breakpoints com legenda
for i, (bp, kv) in enumerate(zip(valid_breakpoints, break_K)):
    if i == 0:
        ax.plot(bp, 0, 'go', markersize=10, label='Ponto de ruptura')
    else:
        ax.plot(bp, 0, 'go', markersize=10)

    ax.annotate(f"s = {bp:.2f}\n$K_p$ = {kv:.2f}",
                xy=(bp, 0), xytext=(bp, 6),
                ha='center', fontsize=9, color='white',
                bbox=dict(boxstyle="round,pad=0.4",
                          fc="darkgreen", alpha=0.6))

# ------ faixa de Routh ------
ax.text(0.45, 0.15,
        f"Faixa estável segundo Routh-Hurwitz: {K_lower0:.1f} < $K_p$ < 0",
        transform=ax.transAxes, fontsize=10,
        va='bottom', ha='right',
        bbox=dict(boxstyle="round,pad=0.3", fc="black", alpha=0.6))

ax.legend(loc='upper left')
ax.set_xlim(-60, 10)
ax.set_ylim(-40, 40)
plt.show()

########################## Parâmetros do Sistema ###############################
# Parâmetros:
rho = 1025 # massa específica da água (kg/m3)
CDl = 0.01 # coeficiente de arrasto dos lemes (fólio fino) (adimensional)
CDc = 1.0 # coeficiente de arrasto 2D do casco (cilindro) (adimensional)
c = 0.026 # corda do fólio (m)
b = 0.052 # envergadura do fólio (m)
m = 16.0 # massa do AUV (kg)
D = 0.15 # diâmetro do AUV (m)
L = 1.2 # comprimento do AUV (m)
GB = D/4 # distância entre G e B (m)
Jyy = (m/2)*(1/12)*(3*((D/2)**2)+(L**2)) + (m/2)*((D/2)**2) # momento de inércia transversal do cilíndro em relação a B (kg.m2)
Jzz = m*(1/12)*(3*((D/2)**2)+(L**2)) # momento de inércia transversal do cilíndro em relação a B (kg.m2)
g = 9.81 # aceleração gravitacional (m/s2)
d = 0.9 *(L/2) # distância do centro do fólio em relação à meia nau (m)
U = 2.0 # velocidade horizontal inicial do AUV (m/s)
m11 = 0.029*rho*math.pi*(D**2)*L*(1/6) # massa adicional de surge (kg)
m33 = 0.96*rho*math.pi*(D**2)*L*(1/4) # massa adicional de heave (kg)
m22 = m33 # massa adcional de sway (kg)
m55 = 0.96*rho*math.pi*(L**3)*(1/12)*(D**2) # momento de inércia adicional de pitch (kg.m2)
m66 = m55 # momento de inércia adicional de yaw (kg.m2)
omega_m = math.radians(10)/1 # velocidade angular máxima de pitch (rad/s)
theta_m = math.radians(15) # ângulo máximo de pitch (graus)
defl = -10 # ângulo de deflexão dos lemes (teste degrau) (graus)
ta = 10 # tempo inicial de atuação do leme (s)
tb = 40 # tempo final de atuação do leme (s)

############################## Modelo Linear ###################################
# Parâmetros de Controle e Saturação:
Kp = -1.0    # ganho proporcional
Td = 1.0    # tempo derivativo
a = np.deg2rad(15)            # amplitude do degrau de referência (constante) (graus)
r = a                         # referência constante
phi_max = np.deg2rad(10.0)  # saturação máxima do leme (graus) (ex.: ±10°)
phi_min = -phi_max
Ahc = np.zeros((3, 3))
Bhc = np.zeros((3, 1))

# Montagem do Sistema:
Ahc[0, 0] = -(((3/8)*U*math.sin(theta_m)*rho*0.82*CDc*D*L + 2*rho*c*U*math.pi*b + 2*0.5*rho*CDl*c*b*U + rho*L*D*U*math.pi)/(m + m22))
Ahc[0, 1] = 0
Ahc[0, 2] = -U*((m + m11)/(m + m22))

Ahc[1, 0] = 0
Ahc[1, 1] = 0
Ahc[1, 2] = 1

Ahc[2, 0] = (2*rho*c*b*U*math.pi*d + 2*0.5*rho*CDl*c*b*U*d - 0.25*rho*(L**2)*D*U*math.pi)/(Jzz + m66)
Ahc[2, 1] = 0
Ahc[2, 2] = -((omega_m*rho*0.80*CDc*D*(L**4))/(Jzz + m66))*(100*(3/4)*(1/64))

Bhc[0, 0] = (2*rho*c*(U**2)*math.pi*b)/(m + m33)
Bhc[1, 0] = 0
Bhc[2, 0] = -(2*rho*c*b*(U**2)*math.pi*d)/(Jzz + m66)

Chc = np.array([[0,1,0]])

# Função para Simular o Sistema Controlável no Espaço de Estados:
def LAUVhc_lin(t, X):
  X = X.reshape(-1,1)        # (n,1)
  y = Chc @ X

  # Atuador
  u = Kp * (r - y)                        # lei proporcional
  u_sat = np.clip(u, phi_min, phi_max)    # saturação...

  X_dot = Ahc @ X + Bhc @ u_sat
  return X_dot.flatten()

############################## Modelo Não Linear ###############################
# Função para Simular o Sistema Não com Controle Linear:
def LAUVhc_non_lin(t, X):
  u, v, psi, psi_dot = X
  #de = - psi_dot # termo derivativo para corrigir regime permanente...

  # Atuador
  u_ctrl = Kp * (r - psi) #+ Kp * Td * de                        # lei proporcional
  u_sat = np.clip(u_ctrl, phi_min, phi_max)    # saturação...

  # ângulos
  beta = np.arctan2(v, u)
  phi = u_sat
  alpha = phi - beta

  # forças no fólio
  Lf = Lift_foil(u, v, alpha)
  Rf = Drag_foil(u, v)
  Lc = Lift_hull(u, v, beta)

  # Equações (resolvidas para derivadas)
  A = np.array([
  [m, 0, 0],
  [0, m + m22, 0],
  [0, 0, (Jzz + m66)]
  ])
  b = np.array([
  m*v*psi_dot,
  2*Lf*np.cos(beta) - 2*Rf*np.sin(beta) - (m+m11)*u*psi_dot - (1/2)*rho*0.82*CDc*D*L*v*np.abs(v) - Lc*np.cos(beta),
  - 2*(Lf*np.cos(beta) - Rf*np.sin(beta))*d - 100*(1/64)*rho*0.80*CDc*D*(L**4)*psi_dot*np.abs(psi_dot) - Lc*(L/4)*np.cos(beta)
  ])

  # Resolve u_dot, w_dot, theta_2dot
  du_dt, dw_dt, dpsi_dot_dt = np.linalg.solve(A, b)
  dpsi_dt = psi_dot

  return np.hstack((du_dt, dw_dt, dpsi_dt, dpsi_dot_dt))

################################### Simulação ##################################
# Configuração Inicial:
t_final = 30
t_eval = np.linspace(0, t_final, 2001)
X0 = [U, 0, 0 ,0] # estado inicial X0 = [u, v, psi, psi_dot]
X0c = [0, 0, 0]   # estado inicial X0c = [v, psi, psi_dot]

# Solver:
solh_lin = solve_ivp(LAUVhc_lin, [0, t_final], X0c, t_eval=t_eval, atol=1e-8, rtol=1e-6)
solh_non_lin = solve_ivp(LAUVhc_non_lin, [0, t_final], X0, t_eval=t_eval, atol=1e-8, rtol=1e-6)

# Reconstrução:
xs = solh_lin.y.T

ys = (Chc @ xs.T).flatten()
u_lin = Kp * (r - ys)
u_sat_lin = np.clip(u_lin, phi_min, phi_max)

u_non_lin = np.zeros(solh_non_lin.t.size)
for i in range(solh_non_lin.t.size):
    psi = solh_non_lin.y[2][i]         # vetor de estado no instante t_i
    u_ctrl = Kp * (r - psi)
    u_sat = np.clip(u_ctrl, -phi_max, phi_max)
    u_non_lin[i] = u_sat

psi_non_lin = solh_non_lin.y[2]
psi_dot_non_lin = solh_non_lin.y[3]

########################## Plotagem dos Resultados #############################
# Ângulo de Rumo:
plt.figure(figsize=(10,6))
plt.text(
    0.70, 0.40,                        # posição (em fração do eixo)
    f"$K_p$ = {Kp:.2f}",               # \n$T_d$ = {Td:.2f}
    transform=plt.gca().transAxes,     # posiciona relativo ao gráfico
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4')
)
plt.plot(solh_lin.t, np.rad2deg(ys), color=ciano_neon, linestyle='--', label="$\psi$(t): Orientação Angular do LAUV - Modelo Linear")
plt.plot(solh_non_lin.t, np.rad2deg(psi_non_lin), color=ciano_neon, linestyle='-', label="$\psi$(t): Orientação Angular do LAUV - Modelo Não Linear")
#plt.plot(solh_lin.t, np.rad2deg(ys), color=ciano_neon, linestyle='--', label="$\psi$(t): Orientação Angular do LAUV - Modelo Linear - Controlador P")
#plt.plot(solh_non_lin.t, np.rad2deg(psi_non_lin), color=ciano_neon, linestyle='-', label="$\psi$(t): Orientação Angular do LAUV - Modelo Não Linear - Controlador PD")
plt.axhline(np.rad2deg(r), color='r', linestyle='--', label=f"ângulo de referência: $\psi$ = {np.rad2deg(r):.1f} graus")
plt.xlabel("Tempo (s)")
plt.ylabel('Ângulo (graus)')
plt.title("Controlador Proporcional de Rumo (Yaw) do LAUV - Movimento no Plano Horizontal")
#plt.title("Controlador de Rumo (Yaw) do LAUV - Movimento no Plano Horizontal")
plt.legend(loc="center right")
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# Atuador:
plt.figure(figsize=(10,6))
plt.text(
    0.75, 0.40,                        # posição (em fração do eixo)
    f"$\phi_{{máx}}$ = {np.rad2deg(phi_max):.1f} graus",
    transform=plt.gca().transAxes,     # posiciona relativo ao gráfico
    fontsize=11,
    verticalalignment='top',
    bbox=dict(facecolor='gray', alpha=0.8, edgecolor='white', boxstyle='round,pad=0.4')
)
plt.plot(solh_lin.t, np.rad2deg(u_sat_lin), color=laranja_neon, linestyle='--', label="$\phi$(t): Ângulo do Leme - Modelo Linear")
plt.plot(solh_non_lin.t, np.rad2deg(u_non_lin), color=laranja_neon, linestyle='-', label="$\phi$(t): Ângulo do Leme - Modelo Não Linear")
#plt.plot(solh_lin.t, np.rad2deg(u_sat_lin), color=laranja_neon, linestyle='--', label="$\phi$(t): Ângulo do Leme - Modelo Linear - Controlador P")
#plt.plot(solh_non_lin.t, np.rad2deg(u_non_lin), color=laranja_neon, linestyle='-', label="$\phi$(t): Ângulo do Leme - Modelo Não Linear - Controlador PD")
plt.xlabel("Tempo (s)")
plt.ylabel("Ângulo (graus)")
plt.title("Ângulo de Deflexão do Leme Saturado")
plt.legend(loc="center right")
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()

# Plano de Fase:
plt.figure(figsize=(10,6))
plt.plot(np.rad2deg(psi_non_lin), np.rad2deg(psi_dot_non_lin), color=vermelho_neon, linestyle='-', label="Trajetória do LAUV no Plano de Fase")
plt.xlabel("Ângulo (graus)")
plt.ylabel('Taxa Angular (graus/s)')
plt.title("Plano de Fase do Modelo Não Linear - Movimento no Plano Horizontal")
plt.legend(loc="upper right")
plt.grid(True)
plt.tight_layout()
mplcyberpunk.add_glow_effects()
plt.show()